In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("..")

import numpy as np
import pandas as pd
from src.predict import preprocess

In [2]:
contoh = pd.DataFrame([{
    "person_age": 30,
    "person_income": 50000,
    "person_home_ownership": "RENT",
    "person_emp_length": np.nan,
    "loan_intent": "EDUCATION",
    "loan_grade": "C",
    "loan_amnt": 10000,
    "loan_int_rate": 13.5,
    "loan_percent_income": 0.20,
    "cb_person_default_on_file": "Y",
    "cb_person_cred_hist_length": 4,
}])

preprocess(contoh)

,person_age,person_income,person_emp_length,loan_amnt,loan_int_rate,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length,person_home_ownership_OTHER,person_home_ownership_OWN,...,loan_intent_HOMEIMPROVEMENT,loan_intent_MEDICAL,loan_intent_PERSONAL,loan_intent_VENTURE,loan_grade_B,loan_grade_C,loan_grade_D,loan_grade_E,loan_grade_F,loan_grade_G
0,0.375194,-0.311575,-0.19488,0.055102,0.804224,0.286376,1,-0.440545,0,0,...,0,0,0,0,0,1,0,0,0,0


In [3]:
import joblib
from pprint import pprint

config = joblib.load("../models/preprocess_config.pkl")
pprint(config)

{'binary_map': {'cb_person_default_on_file': {'N': 0, 'Y': 1}},
 'medians': {'loan_int_rate': np.float64(10.99),
             'person_emp_length': np.float64(4.0)},
 'one_hot_columns': ['person_home_ownership', 'loan_intent', 'loan_grade'],
 'scale_columns': ['person_age',
                   'person_income',
                   'person_emp_length',
                   'loan_amnt',
                   'loan_int_rate',
                   'loan_percent_income',
                   'cb_person_cred_hist_length']}


In [4]:
scoring = joblib.load("../models/scoring_config.pkl")
pprint(scoring)

{'class_weight_W': 3.641208791208791,
 'cost_fn': 500,
 'cost_fp': 50,
 'threshold': 0.24,
 'tier_bins': [0, 0.24, 0.5, 0.8, 1.0],
 'tier_labels': ['Low Risk', 'Medium Risk', 'High Risk', 'Very High Risk']}


In [5]:
from src.predict import predict_risk

predict_risk(contoh)

,skor_risiko,perkiraan_peluang,risk_tier,keputusan,tindakan
0,0.0,0.073,Low Risk,Proses,Proses otomatis / jalur cepat


In [6]:
contoh2 = contoh.copy()
contoh2["loan_percent_income"] = 0.60
contoh2["loan_grade"] = "F"

pd.concat([predict_risk(contoh), predict_risk(contoh2)])

,skor_risiko,perkiraan_peluang,risk_tier,keputusan,tindakan
0,0.0,0.073,Low Risk,Proses,Proses otomatis / jalur cepat
0,1.0,0.990,Very High Risk,Tinjau,"Tolak, atau setujui hanya dengan jaminan"


In [7]:
X_test = pd.read_parquet("../data/processed/X_test.parquet")
pred = pd.read_parquet("../data/processed/predictions.parquet")
raw = pd.read_parquet("../data/processed/credit_risk_dataset_clean.parquet")

# ambil peminjam yang sama dengan data test, dalam bentuk mentah, tanpa jawaban
raw_test = raw.loc[X_test.index].drop(columns="loan_status")
print(raw_test.shape)

(6336, 11)


In [8]:
from src.predict import preprocess

hasil_pre = preprocess(raw_test)

print("Kolom sama:", list(hasil_pre.columns) == list(X_test.columns))
print("Isi sama:", np.allclose(hasil_pre.astype(float), X_test.astype(float)))

Kolom sama: True
Isi sama: True


In [9]:
from src.predict import model

new_score = model.predict_proba(hasil_pre)[:,1]
print("Skor sama:", np.allclose(new_score, pred["cb_prob"]))
print("Selisih terbesar:", np.abs(new_score - pred["cb_prob"].to_numpy()).max())

Skor sama: True
Selisih terbesar: 0.0


In [10]:
from src.predict import predict_risk

hasil = predict_risk(raw_test)
print(hasil["risk_tier"].value_counts().reindex(["Low Risk", "Medium Risk", "High Risk", "Very High Risk"]))
print()
print(hasil["keputusan"].value_counts())

risk_tier
Low Risk          3546
Medium Risk       1453
High Risk          346
Very High Risk     991
Name: count, dtype: int64

keputusan
Proses    3546
Tinjau    2790
Name: count, dtype: int64


In [11]:
beda = (hasil_pre.astype(float) - X_test.astype(float)).abs().max()
beda[beda > 1e-6]

Series([], dtype: float64)

In [12]:
print(raw_test["cb_person_default_on_file"].unique())

<ArrowStringArray>
['N', 'Y']
Length: 2, dtype: str
